# Reproducing Gerken, Darcher et al. (2025) from dandiset 001616

Paper: Gerken F, Darcher A, Gonçalves PJ, Rapp R, Elezi I, Niediek J, Kehl MS, Reber TP, Liebe S,
Macke JH, Mormann F, Leal-Taixé L (2025). Decoding movie content from neuronal population activity in
the human medial temporal lobe. eLife reviewed preprint 106758, version 1.
[doi:10.7554/eLife.106758.1](https://doi.org/10.7554/eLife.106758.1). This attempt used the full
text, which is open access, and the supplementary file with Tables S1 and S2.

Dandiset: [DANDI:001616](https://dandiarchive.org/dandiset/001616/0.260702.0824), version
0.260702.0824 (SUMMER). The paper cites no DANDI version; its data statement says the data will be
published separately. This is the most recent published version. The earlier published version,
0.260519.1133, holds the same 29 files (same blobs) and 28 older copies of the same recordings with
identical spike times (shown in the data check), so the analysis would read the same data from
either; the pinned version holds each recording once.

Code read: the dataset's repository [mormannlab/SUMMER](https://github.com/mormannlab/SUMMER) at
commit 59fc242 (Zenodo 10.5281/zenodo.21165561 archives it). It holds the NWB conversion, the
responsive-unit and cluster code behind Figure 2, plotting notebooks that hold the paper's LSTM
results as numbers, and a simple linear training baseline. It does not hold the paper's LSTM or its
hyperparameters. The fold layout, the grouping of hippocampal subregions and the onset rule here
follow it, as stated in `summer.py`.

No data from outside the dandiset is used. Each claim's section has its own random generator, and
the logistic regression is seeded. The notebook takes about 25 minutes, most of it in the logistic
regression fits. Three companion notebooks use the same helper module: `top_ranked_units.ipynb`
(fig7a and fig7b), and `responsive_zeta_cuts.ipynb` and `responsive_zeta_labels.ipynb`, which run
the ZETA test of the dataset's code for every unit as a variant of the responsive-unit claims.

| Claim | Status |
|---|---|
| dataset-composition | tested |
| fig2b-phc-camera-cuts | tested; blocked, the responsive test is not calibrated |
| tables1-camera-cuts-regions | tested; blocked, depends on the responsive test |
| fig2b-phc-scene-cuts | tested; blocked, the responsive test is not calibrated |
| tables1-few-semantic-responses | tested; blocked, the responsive test is not calibrated |
| tables1-no-indoor-response | tested; blocked, the responsive test is not calibrated |
| results-cluster-regions | tested; blocked, depends on the responsive test |
| fig3b-single-neuron-threshold | tested |
| fig3c-logreg-characters-location | tested |
| fig3c-logreg-cuts-lower | tested |
| fig3e-temporal-gap | tested, with the logistic regression in place of the LSTM |
| fig4-patients-below-pooled | tested |
| fig4-named-patients | tested, assuming subject ids are the paper's patient numbers |
| fig5-region-decoding | tested, with the logistic regression in place of the LSTM |
| fig5-faces-presence | tested, with the logistic regression in place of the LSTM |
| fig6-responsive-subsets | tested, with the logistic regression; blocked, depends on the responsive test |
| fig7b-top500-overlap | tested, in `top_ranked_units.ipynb` |
| fig7a-500-neurons | tested, with the logistic regression in place of the LSTM, in `top_ranked_units.ipynb` |
| fig3c-lstm-decoding | blocked: the LSTM's code and hyperparameters are not published |
| fig3d-temporal-shuffle | blocked: as above |

In [ ]:
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "1")
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import summer

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started_all = started = time.time()
sessions = summer.load_all()
units = summer.unit_table(sessions)
spikes = summer.movie_spikes(sessions)
print(f"loaded {len(sessions)} files, {len(units)} units, from dandiset {summer.DANDISET_ID} "
      f"version {summer.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
# Versions: what the pinned version holds against the earlier published version.
pinned = summer.asset_urls(summer.DANDISET_VERSION)
earlier = summer.asset_urls("0.260519.1133")
blob = {p: b for p, b, _ in pinned}
same = [p for p, b, _ in earlier if blob.get(p) == b]
extra = [(p, u) for p, b, u in earlier if p not in blob]
print(f"version {summer.DANDISET_VERSION}: {len(pinned)} NWB files; version 0.260519.1133: {len(earlier)} NWB files")
print(f"files of 0.260519.1133 also in the pinned version, same path and blob: {len(same)} of {len(pinned)}")
print(f"files only in 0.260519.1133: {len(extra)}, named like {extra[0][0]}")

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(8) as pool:
    old = list(pool.map(lambda a: summer.unit_summary(a[1]), extra))
rows = []
for o in old:
    s = next(s for s in sessions if s["subject_id"] == o["subject_id"])
    new_spikes = np.sort(np.concatenate(s["spikes_ms"]))
    same_n = len(new_spikes) == len(o["spikes_ms"])
    rows.append({"patient": int(o["subject_id"]), "units_old": o["n_units"], "units_pinned": len(s["units"]),
                 "spikes_old": len(o["spikes_ms"]), "spikes_pinned": len(new_spikes),
                 "same_spike_times": bool(same_n and np.allclose(new_spikes, o["spikes_ms"], atol=1e-3)),
                 "same_regions": sorted(o["regions"]) == sorted(s["units"].brain_region)})
old_cmp = pd.DataFrame(rows).sort_values("patient")
print(old_cmp.to_string(index=False))
print("patients of the pinned version without an older copy:",
      sorted(set(int(s["subject_id"]) for s in sessions) - set(old_cmp.patient)))
print("older copies with the same spike times as the pinned file:", int(old_cmp.same_spike_times.sum()), "of", len(old_cmp))

The earlier published version holds the 29 files of the pinned version unchanged and 28 more,
named `ses-1_behavior+ecephys`, one per patient except patient 3. Each of these holds the same
units, regions and spike times as the pinned file of its patient, so that version deposits 28
recordings twice. The pinned version holds each recording once.

In [ ]:
pat = pd.DataFrame([{"patient": int(s["subject_id"]), "sex": s["sex"], "age": s["age"], "units": len(s["units"]),
                     "single_units": int(s["units"].is_single_unit.sum())} for s in sessions])
ages = pat.age.str.extract(r"P(\d+)Y")[0].astype(int)
print(f"patients {len(pat)}; female {int((pat.sex == 'F').sum())}, male {int((pat.sex == 'M').sum())}; "
      f"ages {ages.min()} to {ages.max()} years (median {ages.median():.0f})")
print(f"units {pat.units.sum()} ({pat.single_units.sum()} marked single units); per patient {pat.units.min()} to {pat.units.max()}")
print("\nUnits by `brain_region` as stored:")
print(units.brain_region.value_counts().to_string())
print("\nUnits by region as grouped here (H = AH, MH, PH; FF, LG and PRC are 'other'):")
print(units.region.value_counts().to_string())
print("\nUnits per patient and region:")
print(pd.crosstab(units.patient, units.region, margins=True).to_string())
mismatch = 0
for s in sessions:
    region_of = dict(zip(s["electrodes"].csc_nr, s["electrodes"].brain_region))
    mismatch += int((s["units"].csc_nr.map(region_of) != s["units"].brain_region).sum())
print("\nunits whose region differs from that of their electrode:", mismatch)
print("units table description:", sessions[0]["units_description"])

29 patients, 17 of them female, and 2286 units, 30 to 137 per patient, as the paper states. The
paper's Results give ages 22 to 63 for these 29 patients; the files give 19 to 62, the range the
Methods give for all 46 patients recorded. The region counts differ from the paper's (A 580, H 794,
EC 440, PHC 373, PIC 99): grouping AH, MH and PH as hippocampus, as the dataset's code does, gives
A 559, H 846, EC 493, PHC 293 and PIC 68, and 27 units lie in regions the paper does not name (FF,
LG, PRC). The totals agree, so the units were assigned to regions differently. Every unit's region
matches that of its electrode. No unit is labelled `H`; the dataset's training code
(`ML_framework/nwb_loading/nwb_loading.py`) compares `brain_region` with `H` when asked for the
hippocampus, so it would select no unit. The units table states its spike times in milliseconds,
where NWB expects seconds.

In [ ]:
rows = []
for s in sessions:
    w = s["cleaned_watchlogs"]
    raw = s["raw_watchlogs"]
    pa = summer.pauses(w)
    sp = np.concatenate(s["spikes_ms"])
    in_pause = sum(int(((sp > a) & (sp < b)).sum()) for a, b in pa)
    rows.append({"patient": int(s["subject_id"]), "frames_logged": len(w),
                 "analysis_frames_covered": float(np.isin(np.round(s["frame_times_analysis"], 2), np.round(w.pts, 2)).mean()),
                 "pts_first_s": w.pts.iloc[0], "pts_last_s": w.pts.iloc[-1],
                 "pauses": len(pa), "pause_min": (pa[:, 1] - pa[:, 0]).sum() / 60000 if len(pa) else 0.0,
                 "frame_interval_ms": float(np.median(np.diff(w.rec_ms[np.r_[True, np.diff(w.pts) != 0]]))),
                 "raw_backward_steps": int((np.diff(raw.pts) < 0).sum()),
                 "spike_min_ms": sp.min(), "spike_max_ms": sp.max(), "rec_last_ms": w.rec_ms.iloc[-1],
                 "spikes_in_pauses": in_pause, "spikes_outside_log": int(((sp < w.rec_ms.iloc[0]) | (sp > w.rec_ms.iloc[-1])).sum())})
tb = pd.DataFrame(rows)
print(tb.round(3).to_string(index=False))
print("\nmedian frame interval on the recording clock, ms:", tb.frame_interval_ms.min().round(2), "to", tb.frame_interval_ms.max().round(2))
print("analysis frames (credits excluded):", len(sessions[0]["frame_times_analysis"]), "from",
      sessions[0]["frame_times_analysis"][0], "to", sessions[0]["frame_times_analysis"][-1], "s; all frames:",
      len(sessions[0]["frame_times_base"]))
print("80 ms bin edges identical in all files:", all(np.array_equal(s["edges_s"], sessions[0]["edges_s"]) for s in sessions),
      "; bins:", len(sessions[0]["edges_s"]) - 1)
print("annotation and indicator tables identical in all files:",
      all(s["annotations_base"].equals(sessions[0]["annotations_base"]) for s in sessions),
      all(all(np.array_equal(s["indicator"][k], sessions[0]["indicator"][k]) for k in sessions[0]["indicator"]) for s in sessions))

# Do the patient-aligned annotations (stated in seconds in the column, ms in the table description)
# follow from the movie-time annotations through the watchlog?
dev = []
for s in sessions:
    b = s["annotations_base"]
    a = s["annotations_patient_aligned"]
    for lab in ("summer", "camera-cuts"):
        bs = b[(b.label == lab) & (b.value == 1)].start.to_numpy()
        as_ = a[(a.label == lab) & (a.value == 1)].start.to_numpy()
        w = s["cleaned_watchlogs"]
        inside = (bs >= w.pts.iloc[0]) & (bs <= w.pts.iloc[-1])
        mapped = summer.movie_to_rec(bs[inside], w)
        nearest = np.abs(as_[None, :] - mapped[:, None]).min(1) if len(as_) else np.array([np.nan])
        dev.append({"patient": int(s["subject_id"]), "label": lab, "base_onsets": int(inside.sum()),
                    "aligned_onsets": len(as_), "median_abs_diff_ms": float(np.median(nearest))})
dev = pd.DataFrame(dev)
print("\nmovie-time onsets mapped through the watchlog against `annotations_patient_aligned` (ms):")
print(dev.groupby("label")[["base_onsets", "aligned_onsets", "median_abs_diff_ms"]].describe().round(1).T.to_string())
print("column description:", repr(sessions[0]["annotations_patient_aligned_unit"]))

Every frame of the analysis portion (36.72 to 4763.16 s, credits excluded) is in every patient's
cleaned watchlog. Playback was paused in 24 patients, for up to 57 minutes in all (patient 1);
spikes in pauses are dropped (up to 298658 in patient 10). Frames advance every 40.68 to 41.79 ms
of the recording clock while the film has 40 ms frames, so the recording clock is in milliseconds
and playback ran 2 to 4% slower than 25 frames per second; spikes are mapped to movie time through
the watchlog, which absorbs this. Spikes span the logged playback in every file, and 15 files also
hold spikes outside it. The patient-aligned annotations agree with the movie-time annotations
mapped through the watchlog to a median of one frame (40.7 to 41.8 ms). Their start and stop
columns are described as seconds but hold milliseconds, as the table description says. In some
patients that table holds more onsets than the film has (up to 1181 camera cuts against 958),
apparently from replayed portions; the analysis uses the movie-time annotations. All files carry
the same annotations, indicator functions and 80 ms bin edges (59080 bins).

In [ ]:
s0 = sessions[0]
fa = s0["frame_times_analysis"]
analysis = (s0["frame_times_base"] >= fa[0]) & (s0["frame_times_base"] <= fa[-1])
rows = []
for name, (label, value) in summer.LABELS.items():
    v = s0["indicator"][label]
    rows.append({"label": name, "dandiset_label": label, "values": str(sorted(np.unique(v).tolist())),
                 "fraction_of_analysis_frames": float(np.mean(v[analysis] == value)),
                 "segments": int(((s0["annotations_base"].label == label) & (s0["annotations_base"].value == value)).sum())})
lab = pd.DataFrame(rows)
print(lab.round(3).to_string(index=False))
print("Paper (Results): Summer 50/50, Tom 80/20, Persons 95/5, McKenzie 10/90 of frames.")
cc = s0["indicator"]["camera-cuts"]
sc = s0["indicator"]["scenes"]
movie_s = s0["frame_times_base"][-1]
print(f"\ncamera cuts: {int(cc.sum())} single-frame events, one every {movie_s / cc.sum():.2f} s (supplement: one every 3 s)")
print(f"scene cuts: {int(sc.sum())}, mean scene length {movie_s / sc.sum():.1f} s (supplement: 36.15 s); "
      f"scene cut frames that are also camera cut frames: {int(((sc == 1) & (cc == 1)).sum())}")
print("frames of indoor-setting by value (1 indoor, 0 outdoor, 99 neither):",
      pd.Series(s0["indicator"]["indoor-setting"][analysis]).value_counts().to_dict())
print("face labels hold only 0 and 1 (the Methods also exclude frames):",
      [sorted(np.unique(s0["indicator"][k]).tolist()) for k in ("summer-faces", "tom-faces")])
print("all 53 labels:", ", ".join(sorted(s0["indicator"])))

The fractions of frames match the paper's descriptions (Summer 0.53, Tom 0.79, Persons 0.95,
McKenzie 0.09 of the analysis frames). The dandiset has 958 camera cuts, one every 5.25 s, where
the supplement says one every 3 s. Its 133 scene cuts give a mean scene of 37.8 s (supplement
36.15 s), and 121 of them fall on a camera cut frame. Indoor/Outdoor is `indoor-setting`: 1 indoor,
0 outdoor, and 99 for 839 analysis frames that are neither. The face labels hold only 0 and 1,
while the Methods define face labels with a third state for excluded frames; those frames cannot be
identified from the dandiset.

In [ ]:
# Duplicates within the pinned version: identical spike trains across units, identical watchlogs.
keys = {}
dupes = []
for i, t in enumerate(spikes):
    if len(t) < 20:
        continue
    k = tuple(np.round(t[:20], 4))
    if k in keys:
        dupes.append((keys[k], i))
    keys.setdefault(k, i)
print("pairs of units with the same first 20 spike times:", len(dupes))
for a, b in dupes:
    print("  ", units.iloc[a][["patient", "unit_id", "brain_region", "n_spikes"]].to_dict(),
          units.iloc[b][["patient", "unit_id", "brain_region", "n_spikes"]].to_dict())
w0 = {int(s["subject_id"]): tuple(np.round(s["cleaned_watchlogs"].rec_ms.to_numpy()[100:110], 3)) for s in sessions}
print("patients sharing a watchlog:", len(w0) - len(set(w0.values())))
print("units with no spike during playback:", int(sum(len(t) == 0 for t in spikes)))

No two units share their first 20 spike times and no two patients share a watchlog, so no
recording appears twice in the pinned version. Two units have no spike during playback.

In [ ]:
# Clock check: does each patient's population change its rate after camera cuts at the annotated
# times, and not at the same number of random times? Index = mean over units with 100 or more
# spikes of |rate 200-600 ms after - rate 400-0 ms before| / unit's mean rate; increases and
# decreases both count, as regions differ in sign (pooled PSTH below).
rng_clock = np.random.default_rng(1)
ann = sessions[0]["annotations_base"]
cuts = summer.onsets(ann, "camera-cuts", 1, min_presence=None)


def cut_index(ts, ev):
    vals = []
    for t in ts:
        if len(t) < 100:
            continue
        rate = len(t) / (t[-1] - t[0])
        after = summer._counts_in(t, ev + 0.2, 0.4).mean() / 0.4
        before = summer._counts_in(t, ev - 0.4, 0.4).mean() / 0.4
        vals.append(abs(after - before) / rate)
    return np.mean(vals)


rows = []
for s in sessions:
    p = int(s["subject_id"])
    ts = [np.sort(spikes[i]) for i in units.index[units.patient == p]]
    lo, hi = s["cleaned_watchlogs"].pts.iloc[0] + 1, s["cleaned_watchlogs"].pts.iloc[-1] - 1
    ev = cuts[(cuts > lo) & (cuts < hi)]
    true = cut_index(ts, ev)
    null = np.array([cut_index(ts, np.sort(rng_clock.uniform(lo, hi, len(ev)))) for _ in range(20)])
    rows.append({"patient": p, "cuts": len(ev), "index_true": true, "index_shift_2s": cut_index(ts, ev + 2.0),
                 "null_mean": null.mean(), "null_sd": null.std(), "z": (true - null.mean()) / null.std()})
clock = pd.DataFrame(rows)
print(clock.round(3).to_string(index=False))
print("patients with z above 3:", int((clock.z > 3).sum()), "of", len(clock), "; not above 3:", clock.patient[clock.z <= 3].tolist())

# Pooled PSTH around camera cuts, by region (rate / unit's mean rate, 100 ms bins from -1 to 1 s).
bins = np.arange(-1.0, 1.0001, 0.1)
ps = {}
for region in ("A", "H", "EC", "PHC"):
    vals = []
    for i in units.index[units.region == region]:
        t = np.sort(spikes[i])
        if len(t) < 100:
            continue
        rate = len(t) / (t[-1] - t[0])
        vals.append(summer._counts_in(t, (cuts[:, None] + bins[None, :-1]).ravel(), 0.1).reshape(len(cuts), -1).mean(0) / 0.1 / rate)
    ps[region] = np.mean(vals, 0)
print("\npooled rate around camera cuts / mean rate, bins starting at (s):")
print(pd.DataFrame(ps, index=np.round(bins[:-1], 1)).T.round(3).to_string())

In 11 of 29 patients the population changes its rate after camera cuts more than after random
times (z above 3). In the other 18 (patients 1, 2, 4, 5, 6, 7, 8, 9, 11, 12, 14, 18, 21, 24, 25, 26,
27 and 28) this check does not establish the alignment, though it does not contradict it. Pooled
over patients, the PHC rate rises to 1.09 times its mean 200 to 300 ms after a cut, and the A and
EC rates fall to 0.95 at 300 to 400 ms. Decoding Summer with its labels shifted by 300 s gives kappa
0.015 against 0.301 with the true labels (fig3c section); per patient, the true labels give the
higher kappa in 18 of 29 patients (fig4 section). The spikes and annotations are consistent with
one clock: shown for the pooled data, and for 11 patients one by one.

## Responsive units

Shared by the next six claims. For every unit and the 11 labels of Table S1, the Methods' test as
fixed in the claims file: onsets preceded by at least 1 s without the label and, for labels with a
duration, lasting at least 1 s; the code's onset rule (no minimum duration) is computed as a
variant. Indoor and Outdoor are the values 1 and 0 of `indoor-setting`.

In [ ]:
# Check of the onset rule on a small table: segments at 5-8 s, 8.5-12 s and 20-21 s give onsets at
# 5 s (first segment, nothing before it) and 20 s; 8.5 s follows the first segment by 0.5 s.
toy = pd.DataFrame({"label": "x", "value": 1, "start": [5.0, 8.5, 20.0], "stop": [8.0, 12.0, 21.0]})
assert list(summer.onsets(toy, "x", 1, min_presence=None)) == [5.0, 20.0]
assert list(summer.onsets(toy, "x", 1, min_presence=2.0)) == [5.0]

LABEL_ORDER = ["Tom", "Summer", "McKenzie", "Persons", "Tom Faces", "Summer Faces", "Summer Presence",
               "Camera Cuts", "Scene Cuts", "Indoor", "Outdoor"]
events, events_code = {}, {}
for name in LABEL_ORDER:
    label, value = summer.LABELS[name]
    events[name] = summer.onsets(ann, label, value, min_presence=None if label in summer.CUT_LABELS else 1.0)
    events_code[name] = summer.onsets(ann, label, value, min_presence=None)
print(pd.DataFrame({"onsets (Methods rule)": {k: len(v) for k, v in events.items()},
                    "onsets (code rule)": {k: len(v) for k, v in events_code.items()}}).to_string())

started = time.time()
jobs = []
for s in sessions:
    p = int(s["subject_id"])
    idx = units.index[units.patient == p].to_numpy()
    rng_ = (s["cleaned_watchlogs"].pts.iloc[0], s["cleaned_watchlogs"].pts.iloc[-1])
    jobs.append((summer.responsive_table, ([spikes[i] for i in idx], rng_, events)))
    jobs.append((summer.responsive_table, ([spikes[i] for i in idx], rng_,
                                            {k: v for k, v in events_code.items() if summer.LABELS[k][0] not in summer.CUT_LABELS})))
out = summer.run_jobs(jobs)
resp_tables = []
for j, s in enumerate(sessions):
    idx = units.index[units.patient == int(s["subject_id"])].to_numpy()
    for k, rule in ((2 * j, "methods"), (2 * j + 1, "code")):
        df = out[k].copy()
        df["unit"] = idx[df.unit.to_numpy()]
        df["rule"] = rule
        resp_tables.append(df)
resp = pd.concat(resp_tables, ignore_index=True)
resp = resp.merge(units[["patient", "region", "brain_region"]], left_on="unit", right_index=True)
print(f"responsive tests done in {time.time() - started:.0f} s; rows {len(resp)}")
P = resp[resp.rule == "methods"].pivot(index="unit", columns="label", values="p")
print("units with p = 1 because they spike in fewer than a third of the pseudo-trials, by label:")
print((resp[resp.rule == "methods"].assign(low=lambda d: d.active < 1 / 3).groupby("label").low.sum()).to_string())


def responsive_counts(pv, alpha):
    r = pv <= alpha
    by_region = r.groupby(units.region).sum()
    return r.sum(), by_region

The Methods' test is checked against chance: the same test with random event times (as many as
there are camera cut onsets, within each patient's playback), for every unit; then with as many
random times as each label has onsets; and on synthetic units firing as stationary Poisson
processes with random events.

In [ ]:
rng_calib = np.random.default_rng(21)
null_rows = []
for s in sessions:
    p = int(s["subject_id"])
    lo, hi = s["cleaned_watchlogs"].pts.iloc[0] + 1, s["cleaned_watchlogs"].pts.iloc[-1] - 1
    ev = np.sort(rng_calib.uniform(lo, hi, len(events["Camera Cuts"])))
    for i in units.index[units.patient == p]:
        null_rows.append({"unit": i, "region": units.region[i], "p": summer.wilcoxon_simes(spikes[i], ev)[0]})
null = pd.DataFrame(null_rows)
print("Methods' test at random times, fraction of units 'responsive':")
print(null.groupby("region").p.agg(**{"alpha_0.001": lambda v: np.mean(v <= 0.001), "alpha_0.01": lambda v: np.mean(v <= 0.01),
                                       "units": "size"}).round(3).to_string())
print("all units:", {a: round(float(np.mean(null.p <= a)), 3) for a in summer.ALPHAS})
null_p = null.set_index("unit").p


def null_by_label(spk, lo, hi, counts, seed):
    r = np.random.default_rng(seed)
    out = {}
    for name, k in counts.items():
        ev = np.sort(r.uniform(lo, hi, k))
        out[name] = [summer.wilcoxon_simes(t, ev)[0] for t in spk]
    return out


# Random times, as many as each label's onsets (Methods rule), every unit; one seed per patient.
counts_ = {name: len(events[name]) for name in LABEL_ORDER}
jobs = []
for s in sessions:
    p = int(s["subject_id"])
    lo, hi = s["cleaned_watchlogs"].pts.iloc[0] + 1, s["cleaned_watchlogs"].pts.iloc[-1] - 1
    jobs.append((null_by_label, ([spikes[i] for i in units.index[units.patient == p]], lo, hi, counts_, 2100 + p)))
out = summer.run_jobs(jobs)
null_lab = pd.concat([pd.DataFrame(o, index=units.index[units.patient == int(s["subject_id"])]) for o, s in zip(out, sessions)])
print("\nMethods' test at random times, as many as each label's onsets, fraction of all units 'responsive':")
print(pd.DataFrame({"onsets": counts_, **{f"alpha_{a}": (null_lab <= a).mean().round(4) for a in summer.ALPHAS}}).to_string())
synth = []
for rate in (2.0, 10.0):
    for n_ev in (40, 114, 884):
        ps_ = []
        for _ in range(100):
            t = np.sort(rng_calib.uniform(0, 5000, rng_calib.poisson(rate * 5000)))
            ps_.append(summer.wilcoxon_simes(t, np.sort(rng_calib.uniform(10, 4990, n_ev)))[0])
        synth.append({"rate_hz": rate, "events": n_ev, "alpha_0.001": np.mean(np.array(ps_) <= 0.001),
                      "alpha_0.01": np.mean(np.array(ps_) <= 0.01)})
print("\nsynthetic Poisson units, 100 each, fraction 'responsive':")
print(pd.DataFrame(synth).to_string(index=False))

## fig2b-phc-camera-cuts

> Over half of the parahippocampal neurons responded to camera cuts (193 of 373, 51.74%).

In [ ]:
phc = units.index[units.region == "PHC"]
cc_p = P.loc[phc, "Camera Cuts"]
print(f"PHC units: {len(phc)} (paper: 373)")
for alpha in summer.ALPHAS:
    n = int((cc_p <= alpha).sum())
    print(f"alpha {alpha}: {n} of {len(phc)} responsive, {100 * n / len(phc):.2f}% (paper: 193 of 373, 51.74%; tolerance 46.57 to 56.91%)")
print("\nThreshold sweep (fraction of PHC units, %):",
      {a: round(100 * float((cc_p <= a).mean()), 2) for a in (1e-4, 1e-3, 1e-2, 5e-2)})

print("random event times (calibration above), PHC units responsive (%):",
      {a: round(100 * float((null_p[phc] <= a).mean()), 2) for a in summer.ALPHAS})
print("\nPer patient, PHC units responsive to camera cuts at alpha 0.001 and 0.01:")
per = pd.DataFrame({"units": cc_p.groupby(units.patient).size(),
                    "resp_0.001": (cc_p <= 0.001).groupby(units.patient).sum(),
                    "resp_0.01": (cc_p <= 0.01).groupby(units.patient).sum()})
per["pct_0.001"] = 100 * per["resp_0.001"] / per.units
print(per.round(1).to_string())

The ZETA test of the dataset's code, shown here for PHC in the first round, is now run for every
unit and the 11 labels of Table S1 in the companion notebook `responsive_zeta.ipynb`, with its own
check at random times.

**Result: blocked (methods_underspecified).** The Methods' test, as fixed in the claims file,
counts 222 of 293 PHC units (75.77%) responsive to camera cuts at alpha 0.001 and 228 (77.82%) at
0.01, against 51.74% in the paper. The same test counts 72.35% of PHC units, and 68.4% of all
units, responsive at random event times (69.2% with as many random times as there are camera cut
onsets, in the per-label check), and every synthetic Poisson unit at 2 Hz with 884 random events.
Its false-positive rate is far from its alpha, so the procedure here is not the one that produced
the paper's counts, and the claim cannot be judged. The ZETA test of the dataset's code
(`responsive_zeta_cuts.ipynb`) is calibrated (none of 400 units at random times at alpha 0.001,
1.0% at 0.01); it finds 35 of 293 PHC units (11.95%) responsive at 0.001 and 67 (22.87%) at 0.01,
well below the paper's 51.74%, so the paper's count does not come from that test as run here either.

## tables1-camera-cuts-regions

> Camera cuts evoked responses in a large share of neurons in every region (amygdala 46.90%,
> entorhinal cortex 45.68%, hippocampus 38.92%).

In [ ]:
paper_cc = {"A": 46.90, "EC": 45.68, "H": 38.92, "PHC": 51.74}
rows = []
for region in ("A", "EC", "H", "PHC", "PIC", "other"):
    idx = units.index[units.region == region]
    for alpha in summer.ALPHAS:
        n = int((P.loc[idx, "Camera Cuts"] <= alpha).sum())
        rows.append({"region": region, "alpha": alpha, "units": len(idx), "responsive": n,
                     "pct": 100 * n / len(idx), "paper_pct": paper_cc.get(region, np.nan)})
cc_reg = pd.DataFrame(rows)
cc_reg["within_10pct"] = (cc_reg.pct - cc_reg.paper_pct).abs() <= 0.1 * cc_reg.paper_pct
print(cc_reg.round(2).to_string(index=False))
tot = {a: int((P["Camera Cuts"] <= a).sum()) for a in summer.ALPHAS}
print("all units:", {a: f"{n} of {len(P)} ({100 * n / len(P):.2f}%)" for a, n in tot.items()}, "(paper: 1025, 44.84%)")
print("\nPer patient fraction responsive at alpha 0.001 (%), by region (median and range over patients with 5 or more units):")
fr = (P["Camera Cuts"] <= 0.001).groupby([units.region, units.patient]).agg(["mean", "size"])
fr = fr[fr["size"] >= 5]
print((100 * fr["mean"]).groupby(level=0).describe()[["count", "min", "50%", "max"]].round(1).to_string())

**Result: blocked (methods_underspecified)**, as it rests on the selection of
fig2b-phc-camera-cuts. The miscalibrated test counts 64 to 76% of the units of every region
responsive (A 73.70%, EC 73.63%, H 64.54%, PHC 75.77% at alpha 0.001), against 39 to 52% in the
paper, and 62 to 73% of the units of each region at random times. The ZETA test of the dataset's
code counts A 3.22%, EC 0.61% and H 1.30% (`responsive_zeta_cuts.ipynb`).

## fig2b-phc-scene-cuts

> Few parahippocampal neurons responded to scene cuts (25 of 373, 6.70%), but the modulation was
> consistent across the responsive neurons (cluster permutation test, p = 0.018).

In [ ]:
rng_scene = np.random.default_rng(23)
psth = {rule: {name: np.vstack(resp[(resp.rule == rule) & (resp.label == name)].set_index("unit").loc[units.index, "psth"])
               for name in resp[resp.rule == rule].label.unique()} for rule in ("methods",)}
psth_trial = {name: np.vstack(resp[(resp.rule == "methods") & (resp.label == name)].set_index("unit").loc[units.index, "psth_trial_z"])
              for name in LABEL_ORDER}
phc_mask = (units.region == "PHC").to_numpy()
sc_p = P.loc[phc, "Scene Cuts"]
for alpha in summer.ALPHAS:
    r = (sc_p <= alpha).to_numpy()
    obs, pc = summer.cluster_test(psth["methods"]["Scene Cuts"][phc_mask], r, rng_scene)
    obs2, pc2 = summer.cluster_test(psth_trial["Scene Cuts"][phc_mask], r, rng_scene)
    print(f"alpha {alpha}: {int(r.sum())} of {len(r)} PHC units responsive ({100 * r.mean():.2f}%; paper 25 of 373, 6.70%, "
          f"tolerance 4.16 to 9.24%); largest cluster sum |t| {obs:.1f}, p = {pc:.3f}; "
          f"with the code's per-trial z-scoring {obs2:.1f}, p = {pc2:.3f}")
print("threshold sweep (%):", {a: round(100 * float((sc_p <= a).mean()), 2) for a in (1e-4, 1e-3, 1e-2, 5e-2)})
print("onsets used:", len(events["Scene Cuts"]))

**Result: blocked (methods_underspecified).** The Methods' test counts 138 of 293 PHC units
(47.10%) responsive to scene cuts at alpha 0.001 and 180 (61.43%) at 0.01, against 6.70%; with as
many random times as there are scene cut onsets it counts 48.25% of all units, so the test is not
calibrated. With this split, the largest cluster has p = 0.055 with the Methods' z-scoring and
p = 0.020 with the code's per-trial z-scoring at alpha 0.001; at 0.01 there is no cluster. The ZETA
test finds 12 of 293 PHC units (4.10%) at alpha 0.001 and 29 (9.90%) at 0.01, either side of the
tolerance of 4.16 to 9.24%; with 1000 resamplings in place of 100 these are 11 and 31
(`responsive_zeta_cuts.ipynb`).

## tables1-few-semantic-responses

> Only a few neurons responded to the onset of semantic features (Summer 46, Tom 216, Persons 6 and
> Outdoor 4 of 2286 neurons), in contrast to camera cuts (1025).

In [ ]:
table_s1 = {"Tom": 216, "Summer": 46, "McKenzie": 0, "Persons": 6, "Tom Faces": 843, "Summer Faces": 634,
            "Summer Presence": 0, "Camera Cuts": 1025, "Scene Cuts": 106, "Indoor": 0, "Outdoor": 4}
Pc = resp[resp.rule == "code"].pivot(index="unit", columns="label", values="p")
rows = []
for name in LABEL_ORDER:
    row = {"label": name, "onsets": len(events[name]), "paper_total": table_s1[name]}
    for alpha in summer.ALPHAS:
        r = P[name] <= alpha
        row[f"total_{alpha}"] = int(r.sum())
        row[f"pct_{alpha}"] = round(100 * float(r.mean()), 2)
        for region in ("A", "EC", "H", "PHC"):
            row[f"{region}_{alpha}"] = int(r[units.region == region].sum())
        if name in Pc:
            row[f"code_rule_total_{alpha}"] = int((Pc[name] <= alpha).sum())
    rows.append(row)
s1 = pd.DataFrame(rows)
print(s1.to_string(index=False))
print("\nTolerance: Summer 46 +- 13.2 (32.8 to 59.2), Tom 216 +- 21.6 (194.4 to 237.6).")
semantic = ["Summer", "Tom", "McKenzie", "Persons", "Indoor", "Outdoor", "Summer Presence"]
for alpha in summer.ALPHAS:
    print(f"alpha {alpha}: largest semantic fraction {s1.set_index('label').loc[semantic, f'pct_{alpha}'].max():.2f}% "
          f"({s1.set_index('label').loc[semantic, f'pct_{alpha}'].idxmax()}); camera cuts {s1.set_index('label').loc['Camera Cuts', f'pct_{alpha}']:.2f}%")
print("\nPer patient, units responsive to Summer and Tom at alpha 0.001:")
print(pd.DataFrame({"units": units.groupby("patient").size(),
                    "Summer": (P["Summer"] <= 0.001).groupby(units.patient).sum(),
                    "Tom": (P["Tom"] <= 0.001).groupby(units.patient).sum()}).T.to_string())

**Result: blocked (methods_underspecified).** The test counts 1013 units responsive to Summer and
1174 to Tom at alpha 0.001 (paper 46 and 216), and 9.93 to 51.36% of all units for each semantic
label. With as many random times as each label has onsets it counts about as many: 44.05% of units
for Summer's 114 (44.31% at the real onsets), 50.26% for Tom's 152 (51.36%), 11.46% for Indoor's 44
(13.08%). The counts are dominated by false positives, so neither the counts nor the contrast with
camera cuts (70.69%) can be judged. The ZETA test of the dataset's code, calibrated at random times
(none of 800 units at alpha 0.001), finds 11 units for Summer and 13 for Tom
(`responsive_zeta_labels.ipynb`), below the paper's 46 and 216.

## tables1-no-indoor-response

> No neuron responded to the onset of indoor scenes.

In [ ]:
for name in ("Indoor", "McKenzie", "Summer Presence"):
    print(name, f"({len(events[name])} onsets):", {a: int((P[name] <= a).sum()) for a in summer.ALPHAS},
          "| code onset rule:", {a: int((Pc[name] <= a).sum()) for a in summer.ALPHAS})
print("smallest Indoor p:", f"{P['Indoor'].min():.2g}", "; units testable for Indoor (spiking in a third of the pseudo-trials):",
      int((resp[(resp.rule == 'methods') & (resp.label == 'Indoor')].active >= 1 / 3).sum()))
print("reference, all units at random times (calibration above), responsive:", {a: f"{100 * float((null_p <= a).mean()):.2f}%" for a in summer.ALPHAS})

**Result: blocked (methods_underspecified).** The test as fixed counts 299 units responsive to
Indoor onsets at alpha 0.001 (810 at 0.01), and 439 for McKenzie and 227 for Summer Presence, where
Table S1 has 0 for each. With 44 random times in place of the Indoor onsets it counts 11.46% of all
units. An absence cannot be judged with a test whose
false-positive rate is this far from its alpha. The ZETA test of the dataset's code finds 2 units
for Indoor at alpha 0.001 (`responsive_zeta_labels.ipynb`); it is not the paper's test.

## results-cluster-regions

> By the cluster permutation test, parahippocampal neurons responded to the largest set of features
> (Summer, Persons, Camera Cuts, Scene Cuts, Outdoor), compared with the hippocampus (Tom, Camera
> Cuts, Outdoor), the amygdala (Camera Cuts) and the entorhinal cortex (none).

In [ ]:
rng_cluster = np.random.default_rng(24)
paper_sig = {"PHC": {"Summer", "Persons", "Camera Cuts", "Scene Cuts", "Outdoor"},
             "H": {"Tom", "Camera Cuts", "Outdoor"}, "A": {"Camera Cuts"}, "EC": set()}
CL = ["Summer", "Tom", "McKenzie", "Persons", "Camera Cuts", "Scene Cuts", "Indoor", "Outdoor"]
rows = []
started = time.time()
for alpha in summer.ALPHAS:
    for region in ("A", "H", "EC", "PHC"):
        m = (units.region == region).to_numpy()
        for name in CL:
            r = (P.loc[units.index[m], name] <= alpha).to_numpy()
            obs, pc = summer.cluster_test(psth["methods"][name][m], r, rng_cluster)
            rows.append({"alpha": alpha, "region": region, "label": name, "responsive": int(r.sum()),
                         "cluster_sum": obs, "p": pc, "significant": pc < 0.05, "paper": name in paper_sig[region]})
cl = pd.DataFrame(rows)
cl["agree"] = cl.significant == cl.paper
print(f"cluster tests in {time.time() - started:.0f} s")
for alpha in summer.ALPHAS:
    t = cl[cl.alpha == alpha]
    print(f"\nalpha {alpha}: cells agreeing with the paper {int(t.agree.sum())} of {len(t)}")
    print(t.pivot(index="label", columns="region", values="p").round(3).to_string())
    print("significant labels per region:", t[t.significant].groupby("region").label.apply(list).to_dict())
    print("responsive units per cell:")
    print(t.pivot(index="label", columns="region", values="responsive").to_string())

**Result: blocked (methods_underspecified)**, as the split into responsive and non-responsive
units comes from the miscalibrated test. With that split, 11 of 32 cells agree with the paper at
alpha 0.001 and 12 at 0.01. Camera cuts give no cluster in any region (p = 1.000), and at alpha
0.001 PHC has 6 significant labels against 7 in H, 6 in A and 3 in EC.

## Decoding data

Samples are the 59080 bins of 80 ms of the analysis portion; each unit's feature is its spike count
from 800 ms before to 800 ms after the bin. The labels are those of the decoding claims.

In [ ]:
edges = sessions[0]["edges_s"]
counts = summer.binned_counts(spikes, edges)
S = summer.sample_windows(spikes, edges)
n = S.shape[0]
DEC = {"Summer": "summer", "Tom": "tom", "McKenzie": "mckenzie", "Persons": "persons",
       "Indoor/Outdoor": "indoor-setting", "Scene Cuts": "scenes", "Camera Cuts": "camera-cuts"}
Y = {name: summer.bin_labels(sessions[0], label) for name, label in DEC.items()}
print("samples x units:", S.shape, "; spikes per unit per bin, median:", float(np.median(counts.mean(1))))
print(pd.DataFrame({name: {"positive": float(np.mean(y == 1)), "negative": float(np.mean(y == 0)),
                           "excluded": float(np.mean((y != 0) & (y != 1)))} for name, y in Y.items()}).round(3).to_string())
for f in range(5):
    tr, va, te = summer.split(n, f)
    gap_tr_te = np.min(np.abs(edges[tr][:, None] - edges[te][None, ::50])) if f == 0 else None
    print(f"fold {f + 1}: train {len(tr)}, validation {len(va)}, test {len(te)}"
          + (f"; smallest distance from a training to a test sample, every 50th test sample checked: {gap_tr_te:.1f} s" if f == 0 else ""))

## fig3b-single-neuron-threshold

> Thresholding the activity of single neurons does not reliably predict whether Summer is on
> screen; most neurons perform near chance, below the population decoder.

In [ ]:
y = Y["Summer"]
folds = [summer.split(n, f) for f in range(5)]
started = time.time()
single = np.array([[summer.threshold_unit(S[:, i], y, va, te) for (tr, va, te) in folds] for i in range(S.shape[1])])
mean_k = single.mean(1)
print(f"single-unit thresholds in {time.time() - started:.0f} s")
print(f"median unit kappa {np.median(mean_k):.4f}; fraction of units below 0.05: {np.mean(mean_k < 0.05):.3f}; "
      f"best unit {mean_k.max():.3f} (patient {units.patient[mean_k.argmax()]}, {units.region[mean_k.argmax()]}); "
      f"units at or above 0.31: {int((mean_k >= 0.31).sum())}")
print("quantiles of unit kappa (5, 25, 50, 75, 95, 99%):", np.round(np.quantile(mean_k, [.05, .25, .5, .75, .95, .99]), 3))
print("by region, median and max:")
print(pd.Series(mean_k).groupby(units.region).agg(["median", "max", "size"]).round(3).to_string())
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(mean_k, bins=60)
ax.axvline(0.31, color="k", ls="--", label="paper LSTM, all units")
ax.set_xlabel("single-unit kappa, Summer (mean of 5 folds)")
ax.set_ylabel("units")
ax.legend()
plt.show()
print(f"plotted range {mean_k.min():.3f} to {mean_k.max():.3f}; all points shown")

**Result: reproduced.** The median unit's kappa for Summer is 0.0076, 89.8% of units are below
0.05, and the best unit reaches 0.276 (patient 7, amygdala). No unit reaches the paper's population
value of 0.31, nor the pooled logistic regression here (0.301).

## fig3c-logreg-characters-location and fig3c-logreg-cuts-lower

> A logistic regression on spike counts decodes characters and location about as well as the LSTM;
> pooled over patients it reaches kappa 0.28 for Summer and 0.31 for Indoor/Outdoor.

> For scene and camera cuts the logistic regression performs below the LSTM, by 0.1 and 0.07, that
> is about kappa 0.10 and 0.11.

In [ ]:
rng_lr = np.random.default_rng(31)
started = time.time()
jobs = [(summer.fit_logreg, (S, Y[name], *folds[f])) for name in DEC for f in range(5)]
fits = dict(zip([(name, f) for name in DEC for f in range(5)], summer.run_jobs(jobs)))
print(f"{len(jobs)} pooled fits in {time.time() - started:.0f} s")
lstm = {"Summer": (0.3086, 0.0551), "Tom": (0.2300, 0.0604), "McKenzie": (0.3162, 0.0815), "Persons": (0.3599, 0.0477),
        "Indoor/Outdoor": (0.3115, 0.0597), "Scene Cuts": (0.2007, 0.0321), "Camera Cuts": (0.1833, 0.0149)}
rows = []
for name in DEC:
    ks = np.array([fits[(name, f)]["kappa"] for f in range(5)])
    perm_p = []
    for f in range(5):
        r = fits[(name, f)]
        null = summer.shuffled_kappas(r["y_test"], r["pred"], rng_lr)
        perm_p.append((1 + np.sum(null >= r["kappa"])) / (1 + len(null)))
    rows.append({"label": name, "kappa_mean": ks.mean(), "kappa_sem": ks.std(ddof=1) / np.sqrt(5),
                 "folds": np.round(ks, 3).tolist(), "C": [fits[(name, f)]["C"] for f in range(5)],
                 "nonzero_units": [int((fits[(name, f)]["coef"] != 0).sum()) for f in range(5)],
                 "max_perm_p": max(perm_p), "lstm_paper": lstm[name][0], "lstm_sem": lstm[name][1]})
lr = pd.DataFrame(rows).set_index("label")
print(lr.round(4).to_string())
print("\npaper, logistic regression: Summer 0.28 (tolerance 0.225 to 0.335), Indoor/Outdoor 0.31 (0.25 to 0.37); "
      "Scene Cuts 0.10 (0.068 to 0.132), Camera Cuts 0.11 (0.095 to 0.125)")
print("thresholds, LSTM less SEM: Tom 0.17, McKenzie 0.235, Persons 0.312")
print("\ntest kappa at each C, mean over folds:")
print(pd.DataFrame({name: {c: np.mean([fits[(name, f)]["kappa_by_C"][c] for f in range(5)]) for c in summer.C_GRID} for name in DEC}).round(3).to_string())

In [ ]:
# Clock check by decoding: the same fits for Summer with the labels shifted by 300 s (circularly).
started = time.time()
shifted = np.roll(Y["Summer"], int(300 / 0.08))
sh = summer.run_jobs([(summer.fit_logreg, (S, shifted, *folds[f])) for f in range(5)])
print(f"Summer, labels shifted by 300 s: kappa {np.mean([r['kappa'] for r in sh]):.3f} "
      f"(folds {np.round([r['kappa'] for r in sh], 3).tolist()}); true labels {lr.loc['Summer', 'kappa_mean']:.3f}; "
      f"{time.time() - started:.0f} s")

**Result: reproduced.** Pooled over patients, the logistic regression gives kappa 0.301 for
Summer (SEM 0.056; paper 0.28) and 0.359 for Indoor/Outdoor (SEM 0.049; paper 0.31), both within
tolerance. Tom (0.218), McKenzie (0.324) and Persons (0.410) reach the LSTM values less their SEM
(0.17, 0.235, 0.312). Every fold is above chance by the test-label shuffle (p = 0.001, the smallest
possible with 1000 shuffles) for Summer, McKenzie, Persons and Indoor/Outdoor; for Tom one fold has
kappa 0.002 (p = 0.45). The value of C chosen on validation varies across folds (0.001 to 0.03);
over the grid, the test kappa ranges from 0.266 to 0.311 for Summer and from 0.349 to 0.393 for
Indoor/Outdoor. With the Summer labels shifted by 300 s the kappa is 0.015.

**Result: reproduced.** The logistic regression gives kappa 0.0771 for Scene Cuts (SEM 0.027) and
0.0961 for Camera Cuts (SEM 0.020), within the tolerances around 0.10 and 0.11 (Camera Cuts by
0.001) and below the LSTM values of 0.20 and 0.18. Camera Cuts are above chance in every fold
(p = 0.001); Scene Cuts are not in one fold (kappa -0.008, p = 0.89).

## fig3e-temporal-gap

> Without temporal gaps between training, validation and test samples, decoding of Summer is
> almost perfect (kappa 0.99), and it falls to 0.31 as the gap grows to 32 s.

In [ ]:
rng_gap = np.random.default_rng(35)
y = Y["Summer"]
sizes = tuple(len(i) for i in folds[0])
started = time.time()
jobs = [(summer.fit_logreg, (S, y, *summer.random_split(n, sizes, rng_gap))) for _ in range(5)]
for gap in (4, 8, 16):
    jobs += [(summer.fit_logreg, (S, y, *summer.split(n, f, gap_s=gap))) for f in range(5)]
out = summer.run_jobs(jobs)
gap_rows = {"random": [r["kappa"] for r in out[:5]]}
for j, gap in enumerate((4, 8, 16)):
    gap_rows[f"{gap} s"] = [r["kappa"] for r in out[5 + 5 * j: 10 + 5 * j]]
gap_rows["32 s"] = [fits[("Summer", f)]["kappa"] for f in range(5)]
paper_gap = {"random": 0.987, "4 s": 0.584, "8 s": 0.539, "16 s": 0.429, "32 s": 0.309}
gt = pd.DataFrame({k: {"kappa_mean": np.mean(v), "sem": np.std(v, ddof=1) / np.sqrt(5), "min": np.min(v), "max": np.max(v),
                       "paper_lstm": paper_gap[k]} for k, v in gap_rows.items()}).T
print(f"{len(jobs)} fits in {time.time() - started:.0f} s")
print(gt.round(3).to_string())
print("(paper LSTM values from the dataset's code, buffer_lengths/comparison_performance.ipynb)")

**Result: reproduced.** With samples assigned to training, validation and test at random, the
logistic regression decodes Summer with kappa 0.948 (5 draws, 0.942 to 0.953). With gaps of 4, 8, 16
and 32 s it gives 0.555, 0.524, 0.451 and 0.301, close to the LSTM values the dataset's code holds
for the same gaps (0.987 random, then 0.584, 0.539, 0.429 and 0.309).

## fig4-patients-below-pooled

> Decoding from a single patient's neurons is lower than from the pooled population; no patient
> matches the pooled performance, for Summer, Scene Cuts or Indoor/Outdoor.

In [ ]:
rng_pat = np.random.default_rng(33)
started = time.time()
PAT_LABELS = ("Summer", "Scene Cuts", "Indoor/Outdoor")
Yp = {k: Y[k] for k in PAT_LABELS} | {"Summer shifted 300 s": shifted}
pats = sorted(units.patient.unique())
out = summer.run_jobs([(summer.fit_folds, (S, units.index[units.patient == p].to_numpy(), Yp, folds)) for p in pats])
pfits = {(p, name, f): r for p, o in zip(pats, out) for (name, f), r in o.items()}
print(f"{len(pfits)} per-patient fits in {time.time() - started:.0f} s")
pt = pd.DataFrame({name: {p: np.mean([pfits[(p, name, f)]["kappa"] for f in range(5)]) for p in pats} for name in Yp})
pt.insert(0, "units", units.groupby("patient").size())
pperm = pd.DataFrame({f"{name} folds p<=0.001": {p: sum(summer.permutation_p(pfits[(p, name, f)], rng_pat) <= 0.001 for f in range(5))
                                                  for p in pats} for name in PAT_LABELS})
print(pt.join(pperm).round(3).to_string())
for name in PAT_LABELS:
    pooled = lr.loc[name, "kappa_mean"]
    print(f"{name}: pooled {pooled:.3f}; best patient {pt[name].idxmax()} at {pt[name].max():.3f}; "
          f"patients at or above pooled: {int((pt[name] >= pooled).sum())}; median patient {pt[name].median():.3f}; "
          f"patients above chance (p <= 0.001) in all 5 folds: {int((pperm[f'{name} folds p<=0.001'] == 5).sum())}")
print("\nClock check per patient: Summer with true labels against labels shifted by 300 s:")
print(f"patients whose true kappa exceeds the shifted one: {int((pt['Summer'] > pt['Summer shifted 300 s']).sum())} of {len(pats)}; "
      f"shifted kappa from {pt['Summer shifted 300 s'].min():.3f} to {pt['Summer shifted 300 s'].max():.3f} "
      f"(median {pt['Summer shifted 300 s'].median():.3f}); true from {pt['Summer'].min():.3f} to {pt['Summer'].max():.3f}")

**Result: reproduced.** No patient reaches the pooled kappa for any of the three labels. For
Summer the best patient is 7 at 0.268 against 0.301 pooled (median patient 0.060); for Scene Cuts
patient 20 at 0.043 against 0.077 (median 0.010); for Indoor/Outdoor patient 2 at 0.256 against
0.359 (median 0.099). By the test-label shuffle, 5 patients are above chance (p = 0.001) in all
five folds for Summer, 5 for Scene Cuts and 12 for Indoor/Outdoor. With the Summer labels shifted
by 300 s, per-patient kappas range from -0.105 to 0.100 (median -0.010).

## fig4-named-patients

> For Summer, patients 7, 10, 15 and 22 exceed kappa 0.2; for Indoor/Outdoor, patients 2, 13, 20
> and 25 exceed 0.2.

In [ ]:
named = {"Summer": [7, 10, 15, 22], "Indoor/Outdoor": [2, 13, 20, 25], "Scene Cuts": [2, 10, 19, 20]}
hits = 0
for name, ps in named.items():
    vals = pt.loc[ps, name]
    above = pt.index[pt[name] > 0.2].tolist()
    if name != "Scene Cuts":
        hits += int((vals > 0.2).sum())
    print(f"{name}: named patients {dict(zip(ps, vals.round(3)))}; all patients above 0.2: {above}; "
          f"top 4 here: {pt[name].nlargest(4).index.tolist()}")
print("named Summer and Indoor/Outdoor pairs above 0.2:", hits, "of 8")
print("(Scene Cuts: the paper says only that these patients show slightly better results)")

**Result: reproduced.** Taking subject ids as the paper's patient numbers, 7 of the 8 named pairs
exceed 0.2: for Summer patients 7 (0.268), 10 (0.227) and 22 (0.241), with 15 at 0.197; for
Indoor/Outdoor patients 2 (0.256), 13 (0.210), 20 (0.217) and 25 (0.201). The four best patients
here are 7, 22, 10 and 15 for Summer and 2, 20, 13 and 19 for Indoor/Outdoor. The four Summer
patients are above chance in all five folds. For Scene Cuts, three of the patients the paper names
(20, 19, 2) are among the four best here; patient 10 (0.025) is fifth, behind patient 22. That the
named patients come out on top supports the identity of the numbering, without proving it.

## fig5-region-decoding

> Decoding from single regions depends on the feature; the amygdala is best and the hippocampus
> worst for Summer, and the parahippocampal cortex is best and the amygdala worst for
> Indoor/Outdoor.

In [ ]:
started = time.time()
REG_LABELS = ("Summer", "Indoor/Outdoor")
REGS = ("A", "H", "EC", "PHC")
out = summer.run_jobs([(summer.fit_folds, (S, units.index[units.region == region].to_numpy(), {k: Y[k]}, folds))
                       for region in REGS for k in REG_LABELS])
rfits = {(region, name, f): r for (region, k), o in zip([(g, k) for g in REGS for k in REG_LABELS], out) for (name, f), r in o.items()}
print(f"{len(rfits)} region fits in {time.time() - started:.0f} s")
rt = pd.DataFrame({name: {region: np.mean([rfits[(region, name, f)]["kappa"] for f in range(5)]) for region in ("A", "H", "EC", "PHC")}
                   for name in REG_LABELS})
rs = pd.DataFrame({name: {region: np.std([rfits[(region, name, f)]["kappa"] for f in range(5)], ddof=1) / np.sqrt(5)
                          for region in ("A", "H", "EC", "PHC")} for name in REG_LABELS})
print("mean kappa:")
print(rt.round(3).to_string())
print("SEM:")
print(rs.round(3).to_string())
rng_reg = np.random.default_rng(34)
print("test-shuffle p per fold:")
for region in REGS:
    for name in REG_LABELS:
        print(f"  {region} {name}: {[round(summer.permutation_p(rfits[(region, name, f)], rng_reg), 4) for f in range(5)]}")
print("units:", units.region.value_counts()[["A", "H", "EC", "PHC"]].to_dict())
print("paper (LSTM): Summer A 0.285, H 0.091, EC 0.191, PHC 0.171 (dataset code); Indoor/Outdoor PHC 0.30, H 0.26, EC 0.15, A 0.10 (Results)")
cond = {"Summer, A highest": rt.Summer.idxmax() == "A", "Summer, H lowest": rt.Summer.idxmin() == "H",
        "Indoor/Outdoor, PHC highest": rt["Indoor/Outdoor"].idxmax() == "PHC", "Indoor/Outdoor, A lowest": rt["Indoor/Outdoor"].idxmin() == "A"}
print(cond, "; holding:", sum(cond.values()), "of 4")

**Result: partially reproduced.** With the logistic regression, Summer is decoded best from the
amygdala (0.291) and worst from the hippocampus (0.068; EC 0.149, PHC 0.096), and Indoor/Outdoor
best from PHC (0.357), then H (0.302). The amygdala is not the lowest for Indoor/Outdoor: A (0.122)
and EC (0.117) differ by less than their SEMs (0.041, 0.052). Three of the four orderings hold. Not
every region's fold is above chance: by the test-label shuffle, one fold each for A Indoor/Outdoor,
EC Summer, EC Indoor/Outdoor and PHC Summer, and two for H Summer, have p above 0.05. The regions
here hold different units from the paper's (PHC 293 against 373).

## fig5-faces-presence

> Face labels decode slightly better than character labels (Summer 0.33, Tom 0.27), and Summer
> Presence (0.27) is decoded best from the amygdala (0.26).

Tested with the logistic regression in place of the LSTM. The dandiset's face labels hold only 0
and 1, so the frames the Methods exclude (a face neither shown nor clearly absent) are negatives here.

In [ ]:
rng_face = np.random.default_rng(38)
FACE = {"Summer Faces": "summer-faces", "Tom Faces": "tom-faces", "Summer Presence": "summer-presence"}
for name, label in FACE.items():
    Y[name] = summer.bin_labels(sessions[0], label)
print(pd.DataFrame({name: {"positive": float(np.mean(Y[name] == 1)), "negative": float(np.mean(Y[name] == 0))}
                    for name in ("Summer", "Summer Faces", "Summer Presence", "Tom", "Tom Faces")}).round(3).to_string())
started = time.time()
keys = [(name, f) for name in FACE for f in range(5)]
ffits = dict(zip(keys, summer.run_jobs([(summer.fit_logreg, (S, Y[name], *folds[f])) for name, f in keys])))
out = summer.run_jobs([(summer.fit_folds, (S, units.index[units.region == region].to_numpy(), {k: Y[k]}, folds))
                       for region in REGS for k in FACE])
frfits = {(region, name, f): r for (region, k), o in zip([(g, k) for g in REGS for k in FACE], out) for (name, f), r in o.items()}
print(f"{len(ffits) + len(frfits)} fits in {time.time() - started:.0f} s")
rows = []
for name in ("Summer", "Summer Faces", "Summer Presence", "Tom", "Tom Faces"):
    fl = [fits[(name, f)] if name in DEC else ffits[(name, f)] for f in range(5)]
    ks = np.array([r["kappa"] for r in fl])
    row = {"label": name, "pooled": ks.mean(), "pooled_sem": ks.std(ddof=1) / np.sqrt(5), "folds": np.round(ks, 3).tolist(),
           "max_perm_p": max(summer.permutation_p(r, rng_face) for r in fl)}
    for region in REGS:
        rk = [(rfits if (region, name, 0) in rfits else frfits).get((region, name, f), {}).get("kappa", np.nan) for f in range(5)]
        row[region] = np.mean(rk)
    rows.append(row)
ft = pd.DataFrame(rows).set_index("label")
print(ft.round(4).to_string())
print("paper (LSTM): Summer 0.31, Summer Faces 0.33, Tom 0.23, Tom Faces 0.27, Summer Presence 0.27 (amygdala 0.26)")
cond = {"a: Summer Faces above Summer": ft.loc["Summer Faces", "pooled"] > ft.loc["Summer", "pooled"],
        "b: Tom Faces above Tom": ft.loc["Tom Faces", "pooled"] > ft.loc["Tom", "pooled"],
        "c: Summer Presence highest in A": ft.loc["Summer Presence", list(REGS)].astype(float).idxmax() == "A"}
print(cond, "; holding:", sum(bool(v) for v in cond.values()), "of 3")

**Result: partially reproduced.** One of the three conditions holds. Summer Presence is decoded
best from the amygdala (0.292, against PHC 0.132, EC 0.110 and H 0.103; pooled 0.234). The face
labels decode worse than the character labels here, not better: Summer Faces 0.217 against Summer
0.301, Tom Faces 0.167 against Tom 0.218. Every fold of the three new labels is above chance
(p = 0.001). The face labels here keep as negatives the frames the Methods exclude, which may lower
their kappa; that cannot be checked, as the dandiset does not mark those frames.

## fig6-responsive-subsets

> Responsive neurons drive decoding of camera cuts but not of Summer; without them, Summer is
> decoded about as well as from all neurons while camera cuts drop the most, and responsive
> neurons alone decode camera cuts about as well as all neurons but Summer less well.

In [ ]:
rng_sub = np.random.default_rng(36)
started = time.time()
keys, jobs = [], []
sets = {}
for name in ("Summer", "Camera Cuts"):
    r = (P[name] <= 0.001).to_numpy()
    sets[(name, "responsive only")] = np.where(r)[0]
    sets[(name, "non-responsive only")] = np.where(~r)[0]
    # Variant: non-responsive units that spike in at least a third of the pseudo-trials (units
    # below that get p = 1 without a test).
    act = (resp[(resp.rule == "methods") & (resp.label == name)].set_index("unit").active.loc[units.index] >= 1 / 3).to_numpy()
    sets[(name, "non-responsive, active only")] = np.where(~r & act)[0]
    print(f"{name}: responsive {int(r.sum())}; non-responsive {int((~r).sum())}, of which active {int((~r & act).sum())} "
          f"and inactive (p = 1 untested) {int((~r & ~act).sum())}")
    if r.sum() <= (~r).sum():
        for d in range(3):
            sets[(name, f"non-responsive, size-matched {d + 1}")] = np.sort(rng_sub.choice(np.where(~r)[0], r.sum(), replace=False))
    else:
        print(f"{name}: {int(r.sum())} responsive and {int((~r).sum())} non-responsive units; no size-matched non-responsive set")
out = summer.run_jobs([(summer.fit_folds, (S, cols, {name: Y[name]}, folds)) for (name, kind), cols in sets.items()])
sfits = {(name, kind, f): o[(name, f)] for ((name, kind), cols), o in zip(sets.items(), out) for f in range(5)}
print(f"{len(sfits)} subset fits in {time.time() - started:.0f} s")
rows = []
for name in ("Summer", "Camera Cuts"):
    rows.append({"label": name, "set": "complete", "units": S.shape[1], "kappa": lr.loc[name, "kappa_mean"], "sem": lr.loc[name, "kappa_sem"]})
    for (nm, kind), cols in sets.items():
        if nm == name:
            ks = [sfits[(name, kind, f)]["kappa"] for f in range(5)]
            rows.append({"label": name, "set": kind, "units": len(cols), "kappa": np.mean(ks), "sem": np.std(ks, ddof=1) / np.sqrt(5)})
sub = pd.DataFrame(rows)
print(sub.round(3).to_string(index=False))
k = sub.set_index(["label", "set"]).kappa
drop_s = k["Summer", "complete"] - k["Summer", "non-responsive only"]
drop_c = k["Camera Cuts", "complete"] - k["Camera Cuts", "non-responsive only"]
cond = {"a: Summer without responsive >= complete - 0.05": k["Summer", "non-responsive only"] >= k["Summer", "complete"] - 0.05,
        "b: Summer responsive only < complete - 0.05": k["Summer", "responsive only"] < k["Summer", "complete"] - 0.05,
        "c: drop without responsive larger for camera cuts": drop_c > drop_s,
        "d: camera cuts responsive only >= complete - 0.05": k["Camera Cuts", "responsive only"] >= k["Camera Cuts", "complete"] - 0.05}
print(f"drop without responsive units: Summer {drop_s:.3f}, Camera Cuts {drop_c:.3f}")
print(f"variant, drop to non-responsive active units: Summer {k['Summer', 'complete'] - k['Summer', 'non-responsive, active only']:.3f}, "
      f"Camera Cuts {k['Camera Cuts', 'complete'] - k['Camera Cuts', 'non-responsive, active only']:.3f}")
print(cond, "; holding:", sum(bool(v) for v in cond.values()), "of 4")

**Result: blocked (methods_underspecified)**, as the responsive sets come from the miscalibrated
test (1013 of 2286 units for Summer and 1616 for Camera Cuts at alpha 0.001). With these sets,
three of the four conditions of the criterion hold: for Summer, 0.294 without responsive units
against 0.301 for all units; for Camera Cuts, 0.115 with responsive units only against 0.096 for
all, and 0.013 without them. Condition (b) does not hold: Summer with responsive units only gives
0.256, above the complete value less 0.05 (0.251). Size-matched non-responsive sets give 0.262 to
0.300 for Summer; none can be drawn for Camera Cuts, which has 670 non-responsive units. Of these
670, 548 spike in fewer than a third of the pseudo-trials and get p = 1 untested. Keeping only the
non-responsive units that spike, Camera Cuts give 0.034 (122 units) and Summer 0.241 (730 units),
drops of 0.062 and 0.060 from all units, so with active units alone the drop is no larger for
camera cuts than for Summer.

## fig7a-500-neurons and fig7b-top500-overlap

Both are in the companion notebook `top_ranked_units.ipynb`, which fits the 20 sub-splits, ranks the
units and decodes Summer from the top-ranked units.

## Blocked claims

fig3c-lstm-decoding and fig3d-temporal-shuffle rest on the paper's LSTM. Its code is not in the
paper's materials, the dataset's GitHub repository or its Zenodo archive (which hold a two-layer
linear baseline), and the Methods say that the number of linear layers, hidden sizes, batch size,
learning rate, weight decay and dropout were chosen for each label without giving them. These claims
are blocked with `methods_underspecified`. Training an LSTM for 700 epochs on 2286 inputs for each
label and fold would also not fit the time limit on a CPU; this was not tried. fig3d shuffles the
order of the 20 time bins of each input, which the logistic regression on window sums cannot see,
so no stand-in tests it. fig5-faces-presence and fig7a-500-neurons, also LSTM results in the paper,
are tested above and in `top_ranked_units.ipynb` with the logistic regression in their place.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| dataset-composition | partially reproduced |
| fig2b-phc-camera-cuts | blocked (methods underspecified) |
| tables1-camera-cuts-regions | blocked (methods underspecified) |
| fig2b-phc-scene-cuts | blocked (methods underspecified) |
| tables1-few-semantic-responses | blocked (methods underspecified) |
| tables1-no-indoor-response | blocked (methods underspecified) |
| results-cluster-regions | blocked (methods underspecified) |
| fig3b-single-neuron-threshold | reproduced |
| fig3c-logreg-characters-location | reproduced |
| fig3c-logreg-cuts-lower | reproduced |
| fig3e-temporal-gap | reproduced |
| fig4-patients-below-pooled | reproduced |
| fig4-named-patients | reproduced |
| fig5-region-decoding | partially reproduced |
| fig5-faces-presence | partially reproduced |
| fig6-responsive-subsets | blocked (methods underspecified) |
| fig7b-top500-overlap | partially reproduced (`top_ranked_units.ipynb`) |
| fig7a-500-neurons | partially reproduced (`top_ranked_units.ipynb`) |
| fig3c-lstm-decoding | blocked (methods underspecified) |
| fig3d-temporal-shuffle | blocked (methods underspecified) |

How the analysis differs from the paper's:

- The paper's LSTM is not run. Its logistic regression, which the paper reports for the
  patient-wise analysis and Figure 3c, is run as the Methods describe, with class weights in place
  of oversampling and C chosen from four values. It stands in for the LSTM in fig3e, fig5
  (regions, faces and presence), fig6 and fig7a.
- The responsive-unit test follows the Methods as written. On this data it is not calibrated
  (68.4% of units responsive at random times at alpha 0.001 with as many events as camera cuts,
  9.5 to 55.5% with as many as each other label's onsets), so the claims that rest on it are
  blocked. The dataset code's ZETA test is calibrated and gives counts far below Table S1 (camera
  cuts 79 units against 1025, Summer 11 against 46), so neither test as run here gives the paper's
  counts.
- Onsets of a label include its first segment, and the decoding windows at the ends of the analysis
  portion count the spikes just outside it; both were changed after the first verification round.
- Regions are the dandiset's `brain_region`, grouped as in the dataset's code; their counts differ
  from the paper's.
- Cut labels are extended to 520 ms after the cut for scene cuts too; the Methods state it for
  camera cuts.

What was found about the dandiset:

- It holds the 2286 units of the 29 patients, the movie annotations and the watchlogs needed for
  the analyses, and the logistic regression results of the paper are recovered from it.
- The units' regions give counts that differ from the paper's (PHC 293 against 373, EC 493 against
  440, PIC 68 against 99, 27 units in other regions), with no mapping between the two.
- Spike times are stored in milliseconds; the patient-aligned annotation columns are described as
  seconds but hold milliseconds.
- The face labels lack the excluded state the Methods use; the face decoding here keeps those frames
  as negatives.
- The earlier published version 0.260519.1133 holds 28 of the recordings twice.
- The dataset's code identifies responsive units with the ZETA test, not the test of the Methods,
  and its training code selects no hippocampal unit.

In [ ]:
print(f"total run time {(time.time() - started_all) / 60:.1f} min")